# Load data

Conversation from famous movies with annotations

In [7]:
import json

path_data = "data/locomo10.json"
def load_data(path_data):
    with open(path_data, "r") as f:
        conversations = json.load(f)
    return conversations

In [8]:
locomo = load_data(path_data)

In [9]:
from dateutil.parser import parse
import spacy

nlp = spacy.load("en_core_web_sm")

def is_date_like(text):
    try:
        parse(text, fuzzy=False)
        return True
    except:
        return False

def detect_datetime_entity(answer):
    doc = nlp(answer)
    for ent in doc.ents:
        if ent.label_ in ["DATE", "TIME"]:
            return True
    return is_date_like(answer)

In [10]:
from tqdm import tqdm

def preprocess(data):
    dataset = []
    for sample in tqdm(data, desc="Processing samples"):
        conversation = sample["conversation"]
        user = conversation["speaker_a"]
        assistant = conversation["speaker_b"]

        clean_conversation = []
        sessions = []
        for key in conversation.keys():
            if key.startswith("session_") and not key.endswith("_date_time"):
                sessions.append(key)

        for session_id in sessions:
            session = conversation[session_id]
            clean_session = []
            for turn in session:
                if turn["speaker"] == user:
                    clean_session.append({"role": "user", "content": turn["text"]})
                elif turn["speaker"] == assistant:
                    clean_session.append({"role": "assistant", "content": turn["text"]})

            clean_conversation.append(clean_session)

        qa = sample["qa"]
        for qa_item in qa:
            if "question" in qa_item and "answer" in qa_item:
                question = qa_item["question"]
                answer = str(qa_item["answer"])
                
                # detect if the answer is a date/time
                if not detect_datetime_entity(answer) and not detect_datetime_entity(question) and not assistant in answer and not assistant in question:
                    dataset.append({
                        "past_interactions": clean_conversation,
                        "question": question,
                        "answer": answer,
                    })
        
    return dataset

In [11]:
dataset = preprocess(locomo)
print(len(dataset))

Processing samples: 100%|██████████| 10/10 [00:05<00:00,  1.80it/s]

457


In [ ]:
from memory.memory import update_ltm
from llm.openai_utils import ask_llm, answer_question

def predict_answer(question, past_interactions):
    ltm = ""
    for session in tqdm(past_interactions, desc="Processing past interactions"):
        while len(session) > 20:
            interactions = session[:2]
            ltm = update_ltm(0, interactions, ltm)
            session = session[2:]

        ltm = update_ltm(0, session, ltm)

    answer = answer_question(question, "", ltm)
    return answer

In [13]:
example = dataset[0]

past_interactions = example["past_interactions"]
question = example["question"]
gold_answer = example["answer"]

predicted_answer = predict_answer(question, past_interactions)

Processing past interactions: 100%|██████████| 19/19 [07:06<00:00, 22.43s/it]


In [14]:
print(f"Question: {question}")
print(f"Gold Answer: {gold_answer}")
print(f"Predicted Answer: {predicted_answer}")

Question: What fields would Caroline be likely to pursue in her educaton?
Gold Answer: Psychology, counseling certification
Predicted Answer: Caroline would likely pursue education in fields related to counseling, psychology, or mental health, given her interest in supporting those in need of psychological help. Additionally, her involvement in art and expressions through different media suggests she may also consider studies in art therapy or related fields.


## Face recognition

In [31]:
import numpy as np
import face_recognition

import os
from PIL import Image
import glob

adam_driver_path = "static/users_images/adam_driver"
#read all images in the directory
img_files = glob.glob(os.path.join(adam_driver_path, "*.jpg"))

/Users/isir_acide/Documents/Context-aware_Conversation/.venv/lib/python3.13/site-packages/face_recognition_models/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [40]:
list_sames = []

for i in range(len(img_files)):

    for j in range(i + 1, len(img_files)):
            img1 = face_recognition.load_image_file(img_files[i])
            img2 = face_recognition.load_image_file(img_files[j])
            encodings1 = face_recognition.face_encodings(img1)
            encodings2 = face_recognition.face_encodings(img2)

            if len(encodings1) > 0 and len(encodings2) > 0:
                distance = np.linalg.norm(encodings1[0] - encodings2[0])
                if distance < 0.6:
                    list_sames.append(True)
                else:
                    list_sames.append(False)
            else:
                print(f"Could not find face encodings for one of the images: {img_files[i]}, {img_files[j]}")

print(list_sames.count(True) / len(list_sames) * 100)

100.0
